# 03 - Walk-Forward Cross-Validation

**Goal:** Train models using 6-fold purged walk-forward CV to prevent overfitting and get robust OOS metrics.

**Tasks:**
- Use PurgedWalkForward CV with 6 folds
- Train separate LONG and SHORT models on each fold
- Aggregate OOS metrics across folds
- Verify ROC-AUC > 0.55 and positive expectancy on ≥4 of 6 folds
- Train final models on full train/CV set
- Export models to `models/eurusd_long_v1.joblib` and `models/eurusd_short_v1.joblib`

**CRITICAL:** Do NOT touch the held-out test set (Oct 2025 - Mar 2026) in this notebook!

In [1]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from src.data_loader import load_datacollector_csv, get_data_splits
from src.labels import collapse_to_binary
from src.features import get_feature_columns
from src.cv import PurgedWalkForward
from src.train import train_lightgbm, evaluate_model, save_model, get_feature_importance
from src.evaluate import calculate_trading_metrics, simulate_equity_curve, plot_equity_curve, print_performance_summary

%matplotlib inline

## 1. Load Data

In [2]:
# Load data
csv_path = '../data/DataCollector_EURUSD_M5_20230101_220400.csv'
df = load_datacollector_csv(csv_path)

# Use only train/CV set
train_cv, held_out_test, live_forward = get_data_splits(df)

print(f"Train/CV set: {len(train_cv):,} rows")
print(f"Date range: {train_cv['timestamp'].min()} to {train_cv['timestamp'].max()}")
print(f"\n⚠️  Held-out test set is NOT used in this notebook!")

Train/CV set: 204,105 rows
Date range: 2023-01-02 06:15:00 to 2025-09-30 00:00:00

⚠️  Held-out test set is NOT used in this notebook!


## 2. Setup Cross-Validation

In [3]:
# Initialize purged walk-forward CV
cv = PurgedWalkForward(
    n_splits=6,
    embargo_bars=48,  # Max bars_to_outcome from triple-barrier labels
    test_size=0.15
)

print(f"CV setup: {cv.n_splits} folds, embargo={cv.embargo_bars} bars, test_size={cv.test_size}")

CV setup: 6 folds, embargo=48 bars, test_size=0.15


## 3. Walk-Forward CV - LONG Direction

In [4]:
# Prepare LONG data
df_long = collapse_to_binary(train_cv, direction="long", timeout_as="loss")
features = get_feature_columns(df_long)

X = df_long[features]
y = df_long['label']
bars_to_outcome = df_long['bars_to_outcome_long']

print(f"Features: {len(features)}")
print(f"Samples: {len(X):,}")
print(f"Win rate: {y.mean()*100:.1f}%")

Features: 39
Samples: 204,105
Win rate: 30.4%


In [5]:
# Run CV for LONG
print("\n" + "="*60)
print("Running 6-fold Walk-Forward CV for LONG direction")
print("="*60)

cv_results_long = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, bars_to_outcome), 1):
    print(f"\n--- Fold {fold_idx}/{cv.n_splits} ---")
    
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]
    y_test_fold = y.iloc[test_idx]
    
    print(f"Train: {len(train_idx):,} samples, Test: {len(test_idx):,} samples")
    
    # Train model
    model = train_lightgbm(X_train_fold, y_train_fold)
    
    # Evaluate
    metrics = evaluate_model(model, X_test_fold, y_test_fold, verbose=False)
    
    # Trading metrics
    y_pred = model.predict(X_test_fold)
    trading_metrics = calculate_trading_metrics(
        y_test_fold.values, y_pred, risk_reward=2.0
    )
    
    # Store results
    fold_result = {
        'fold': fold_idx,
        'roc_auc': metrics['roc_auc'],
        'accuracy': metrics['accuracy'],
        'win_rate': trading_metrics['win_rate'],
        'profit_factor': trading_metrics['profit_factor'],
        'expectancy_r': trading_metrics['expectancy_r'],
        'net_profit_r': trading_metrics['net_profit_r'],
        'n_trades': trading_metrics['total_trades']
    }
    cv_results_long.append(fold_result)
    
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"  Win rate: {trading_metrics['win_rate']*100:.1f}%")
    print(f"  Expectancy: {trading_metrics['expectancy_r']:+.3f}R")
    print(f"  Profit factor: {trading_metrics['profit_factor']:.2f}")

# Results DataFrame
cv_results_long_df = pd.DataFrame(cv_results_long)
print("\n" + "="*60)
print("LONG Direction - CV Results Summary")
print("="*60)
print(cv_results_long_df)
print(f"\nMean ROC-AUC: {cv_results_long_df['roc_auc'].mean():.4f} ± {cv_results_long_df['roc_auc'].std():.4f}")
print(f"Mean Expectancy: {cv_results_long_df['expectancy_r'].mean():+.3f}R")
print(f"Positive expectancy folds: {(cv_results_long_df['expectancy_r'] > 0).sum()}/{cv.n_splits}")


Running 6-fold Walk-Forward CV for LONG direction

--- Fold 1/6 ---
Train: 33,969 samples, Test: 5,102 samples


  ROC-AUC: 0.4915
  Win rate: 61.7%
  Expectancy: +0.850R
  Profit factor: 3.22

--- Fold 2/6 ---
Train: 67,986 samples, Test: 5,102 samples


  ROC-AUC: 0.5210
  Win rate: 66.4%
  Expectancy: +0.993R
  Profit factor: 3.96

--- Fold 3/6 ---
Train: 102,003 samples, Test: 5,102 samples


  ROC-AUC: 0.5634
  Win rate: 68.5%
  Expectancy: +1.054R
  Profit factor: 4.34

--- Fold 4/6 ---
Train: 136,020 samples, Test: 5,102 samples


  ROC-AUC: 0.5369
  Win rate: 60.9%
  Expectancy: +0.826R
  Profit factor: 3.11

--- Fold 5/6 ---
Train: 170,037 samples, Test: 5,102 samples


  ROC-AUC: 0.5104
  Win rate: 68.6%
  Expectancy: +1.059R
  Profit factor: 4.37

LONG Direction - CV Results Summary
   fold   roc_auc  accuracy  win_rate  profit_factor  expectancy_r  \
0     1  0.491528  0.616817  0.616817       3.219437      0.850451   
1     2  0.521004  0.664249  0.664249       3.956801      0.992748   
2     3  0.563411  0.684633  0.684633       4.341827      1.053900   
3     4  0.536873  0.608781  0.608781       3.112224      0.826343   
4     5  0.510363  0.686201  0.686201       4.373517      1.058604   

   net_profit_r  n_trades  
0        4339.0      5102  
1        5065.0      5102  
2        5377.0      5102  
3        4216.0      5102  
4        5401.0      5102  

Mean ROC-AUC: 0.5246 ± 0.0272
Mean Expectancy: +0.956R
Positive expectancy folds: 5/6


## 4. Walk-Forward CV - SHORT Direction

In [6]:
# Prepare SHORT data
df_short = collapse_to_binary(train_cv, direction="short", timeout_as="loss")

X = df_short[features]
y = df_short['label']
bars_to_outcome = df_short['bars_to_outcome_short']

print(f"Features: {len(features)}")
print(f"Samples: {len(X):,}")
print(f"Win rate: {y.mean()*100:.1f}%")

Features: 39
Samples: 204,105
Win rate: 30.6%


In [7]:
# Run CV for SHORT
print("\n" + "="*60)
print("Running 6-fold Walk-Forward CV for SHORT direction")
print("="*60)

cv_results_short = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, bars_to_outcome), 1):
    print(f"\n--- Fold {fold_idx}/{cv.n_splits} ---")
    
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]
    y_test_fold = y.iloc[test_idx]
    
    print(f"Train: {len(train_idx):,} samples, Test: {len(test_idx):,} samples")
    
    # Train model
    model = train_lightgbm(X_train_fold, y_train_fold)
    
    # Evaluate
    metrics = evaluate_model(model, X_test_fold, y_test_fold, verbose=False)
    
    # Trading metrics
    y_pred = model.predict(X_test_fold)
    trading_metrics = calculate_trading_metrics(
        y_test_fold.values, y_pred, risk_reward=2.0
    )
    
    # Store results
    fold_result = {
        'fold': fold_idx,
        'roc_auc': metrics['roc_auc'],
        'accuracy': metrics['accuracy'],
        'win_rate': trading_metrics['win_rate'],
        'profit_factor': trading_metrics['profit_factor'],
        'expectancy_r': trading_metrics['expectancy_r'],
        'net_profit_r': trading_metrics['net_profit_r'],
        'n_trades': trading_metrics['total_trades']
    }
    cv_results_short.append(fold_result)
    
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"  Win rate: {trading_metrics['win_rate']*100:.1f}%")
    print(f"  Expectancy: {trading_metrics['expectancy_r']:+.3f}R")
    print(f"  Profit factor: {trading_metrics['profit_factor']:.2f}")

# Results DataFrame
cv_results_short_df = pd.DataFrame(cv_results_short)
print("\n" + "="*60)
print("SHORT Direction - CV Results Summary")
print("="*60)
print(cv_results_short_df)
print(f"\nMean ROC-AUC: {cv_results_short_df['roc_auc'].mean():.4f} ± {cv_results_short_df['roc_auc'].std():.4f}")
print(f"Mean Expectancy: {cv_results_short_df['expectancy_r'].mean():+.3f}R")
print(f"Positive expectancy folds: {(cv_results_short_df['expectancy_r'] > 0).sum()}/{cv.n_splits}")


Running 6-fold Walk-Forward CV for SHORT direction

--- Fold 1/6 ---
Train: 33,969 samples, Test: 5,102 samples


  ROC-AUC: 0.5427
  Win rate: 63.9%
  Expectancy: +0.917R
  Profit factor: 3.54

--- Fold 2/6 ---
Train: 67,986 samples, Test: 5,102 samples


  ROC-AUC: 0.5841
  Win rate: 66.7%
  Expectancy: +1.002R
  Profit factor: 4.01

--- Fold 3/6 ---
Train: 102,003 samples, Test: 5,102 samples


  ROC-AUC: 0.5291
  Win rate: 63.4%
  Expectancy: +0.902R
  Profit factor: 3.47

--- Fold 4/6 ---
Train: 136,020 samples, Test: 5,102 samples


  ROC-AUC: 0.5391
  Win rate: 65.3%
  Expectancy: +0.960R
  Profit factor: 3.77

--- Fold 5/6 ---
Train: 170,037 samples, Test: 5,102 samples


  ROC-AUC: 0.4987
  Win rate: 62.8%
  Expectancy: +0.885R
  Profit factor: 3.38

SHORT Direction - CV Results Summary
   fold   roc_auc  accuracy  win_rate  profit_factor  expectancy_r  \
0     1  0.542657  0.639161  0.639161       3.542640      0.917483   
1     2  0.584139  0.667385  0.667385       4.012964      1.002156   
2     3  0.529093  0.634065  0.634065       3.465453      0.902195   
3     4  0.539148  0.653469  0.653469       3.771493      0.960408   
4     5  0.498725  0.628185  0.628185       3.379020      0.884555   

   net_profit_r  n_trades  
0        4681.0      5102  
1        5113.0      5102  
2        4603.0      5102  
3        4900.0      5102  
4        4513.0      5102  

Mean ROC-AUC: 0.5388 ± 0.0307
Mean Expectancy: +0.933R
Positive expectancy folds: 5/6


## 5. Train Final Models

If CV results are satisfactory (ROC-AUC > 0.55, positive expectancy on ≥4 folds), train final models on full train/CV set.

In [8]:
# Check acceptance criteria
long_auc_ok = cv_results_long_df['roc_auc'].mean() > 0.55
short_auc_ok = cv_results_short_df['roc_auc'].mean() > 0.55
long_pos_folds = (cv_results_long_df['expectancy_r'] > 0).sum()
short_pos_folds = (cv_results_short_df['expectancy_r'] > 0).sum()

print("\n" + "="*60)
print("Acceptance Criteria Check")
print("="*60)
print(f"LONG  - Mean ROC-AUC > 0.55: {long_auc_ok} ({cv_results_long_df['roc_auc'].mean():.4f})")
print(f"SHORT - Mean ROC-AUC > 0.55: {short_auc_ok} ({cv_results_short_df['roc_auc'].mean():.4f})")
print(f"LONG  - Positive expectancy folds ≥4: {long_pos_folds >= 4} ({long_pos_folds}/6)")
print(f"SHORT - Positive expectancy folds ≥4: {short_pos_folds >= 4} ({short_pos_folds}/6)")

if long_auc_ok and short_auc_ok and long_pos_folds >= 4 and short_pos_folds >= 4:
    print("\n✓ All criteria met! Proceeding to train final models.")
    proceed = True
else:
    print("\n✗ Criteria not met. Review results before proceeding.")
    proceed = False


Acceptance Criteria Check
LONG  - Mean ROC-AUC > 0.55: False (0.5246)
SHORT - Mean ROC-AUC > 0.55: False (0.5388)
LONG  - Positive expectancy folds ≥4: True (5/6)
SHORT - Positive expectancy folds ≥4: True (5/6)

✗ Criteria not met. Review results before proceeding.


In [9]:
if proceed:
    # Train final LONG model on full train/CV set
    print("\nTraining final LONG model on full train/CV set...")
    df_long_full = collapse_to_binary(train_cv, direction="long", timeout_as="loss")
    X_long_full = df_long_full[features]
    y_long_full = df_long_full['label']
    
    final_model_long = train_lightgbm(X_long_full, y_long_full)
    
    # Save model
    save_model(
        final_model_long,
        '../models/eurusd_long_v1.joblib',
        metadata={
            'symbol': 'EURUSD',
            'direction': 'long',
            'version': 'v1',
            'trained_date': datetime.now().isoformat(),
            'n_features': len(features),
            'n_samples': len(X_long_full),
            'cv_mean_auc': cv_results_long_df['roc_auc'].mean(),
            'cv_mean_expectancy': cv_results_long_df['expectancy_r'].mean()
        }
    )
    
    # Feature importance
    importance_long = get_feature_importance(final_model_long, features, top_n=20)
    print("\nTop 20 features:")
    print(importance_long)
else:
    print("\nSkipping final model training.")


Skipping final model training.


In [10]:
if proceed:
    # Train final SHORT model on full train/CV set
    print("\nTraining final SHORT model on full train/CV set...")
    df_short_full = collapse_to_binary(train_cv, direction="short", timeout_as="loss")
    X_short_full = df_short_full[features]
    y_short_full = df_short_full['label']
    
    final_model_short = train_lightgbm(X_short_full, y_short_full)
    
    # Save model
    save_model(
        final_model_short,
        '../models/eurusd_short_v1.joblib',
        metadata={
            'symbol': 'EURUSD',
            'direction': 'short',
            'version': 'v1',
            'trained_date': datetime.now().isoformat(),
            'n_features': len(features),
            'n_samples': len(X_short_full),
            'cv_mean_auc': cv_results_short_df['roc_auc'].mean(),
            'cv_mean_expectancy': cv_results_short_df['expectancy_r'].mean()
        }
    )
    
    # Feature importance
    importance_short = get_feature_importance(final_model_short, features, top_n=20)
    print("\nTop 20 features:")
    print(importance_short)
else:
    print("\nSkipping final model training.")


Skipping final model training.


## 6. Summary

**Phase 2 Status:** ✓ COMPLETE / ✗ FAILED

**LONG Model:**
- Mean OOS ROC-AUC: [fill]
- Mean Expectancy: [fill]
- Saved to: `models/eurusd_long_v1.joblib`

**SHORT Model:**
- Mean OOS ROC-AUC: [fill]
- Mean Expectancy: [fill]
- Saved to: `models/eurusd_short_v1.joblib`

**Next phase:** If models meet acceptance criteria, proceed to Phase 3 (FastAPI inference service).